### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 0.1_cast_splitting_to_netcdf
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) & Bàrbara Barceló-Llull
### Data from FaSt-SWOT experiment

# 
**DESCRIPTION**:
 This script reads raw MVP (Moving Vessel Profiler) data files (.m1 format), 
 parses their internal metadata (NMEA coordinates, date, time), and splits 
 the continuous data stream into individual 'downcast' and 'upcast' profiles.
 Finally, it exports the separated profiles as NetCDF (.nc) files, ready for 
 further quality control and processing. (modified from a code to read MVP 
 data developed by Eugenio Cutolo)
#
 INPUT: Directory containing raw .m1 files.
#
 OUTPUT: Directory containing parsed _downcast.nc and _upcast.nc files.
### ==============================================================================

In [1]:

import pandas as pd
import xarray as xr
import numpy as np
import os
import re
from glob import glob
from datetime import datetime

# --- CONFIGURATION ---
RAW_DIR_LEG1 = r'C:\Users\ASUS\Desktop\MVP\MVP_fastswot_m1\Leg1'
RAW_DIR_LEG2 = r'C:\Users\ASUS\Desktop\MVP\MVP_fastswot_m1\Leg2'

OUT_DIR_LEG1 = r'C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\raw'
OUT_DIR_LEG2 = r'C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg2\raw'

os.makedirs(OUT_DIR_LEG1, exist_ok=True)
os.makedirs(OUT_DIR_LEG2, exist_ok=True)


# --- 1. HEADER PARSING (METADATA EXTRACTION) ---

def parse_nmea_coord(nmea_str, coord_type='lat'):
    """
    Converts NMEA string format (ddmm.mmmm) to Decimal Degrees.
    Returns np.nan if parsing fails.
    """
    try:
        parts = nmea_str.strip().split(',')
        val_str = parts[0]
        hemi = parts[1] if len(parts) > 1 else ''
        val = float(val_str)
        
        deg = int(val / 100)
        mins = val % 100
        decimal = deg + (mins / 60.0)
        
        # Apply negative sign for South and West hemispheres
        if hemi in ['S', 'W']: 
            decimal = -decimal
        return decimal
    except: 
        return np.nan

def parse_m1_header(filepath):
    """
    Reads and extracts actual metadata from the .m1 file header.
    Looks for Latitude, Longitude, Date, and Time.
    """
    meta = {'lat': np.nan, 'lon': np.nan, 'datetime': None}
    
    with open(filepath, 'r', encoding='latin-1', errors='ignore') as f:
        # Read the first 50 lines where the header is located
        header_lines = [f.readline() for _ in range(50)]
        
    date_str, time_str = None, None
    
    for line in header_lines:
        line = line.strip()
        if "LAT" in line and ":" in line:
            meta['lat'] = parse_nmea_coord(line.split(':')[1], 'lat')
        if "LON" in line and ":" in line:
            meta['lon'] = parse_nmea_coord(line.split(':')[1], 'lon')
        if "Date" in line and ":" in line and "PC" not in line:
            date_str = line.split(':')[1].strip()
        if "Time" in line and ":" in line and "PC" not in line:
            time_str = line.split(':', 1)[1].strip().replace('|', ':')
            
    # Combine date and time strings into a datetime object
    if date_str and time_str:
        try:
            meta['datetime'] = pd.to_datetime(f"{date_str} {time_str}", dayfirst=True)
        except: 
            pass
            
    return meta


# --- 2. GENERIC NETCDF GENERATOR ---

def create_mvp_dataset(df, meta, filename, cast_type):
    """
    Creates a standardized xarray.Dataset object.
    Parameters:
    - df: DataFrame containing the profile data.
    - meta: Dictionary with extracted metadata.
    - filename: Original source filename.
    - cast_type: String indicating 'downcast' or 'upcast'.
    """
    if df.empty: 
        return None

    # Define the core dataset structure
    ds = xr.Dataset(
        data_vars={
            'pressure': (('scan',), df['pressure'].values),
            't1': (('scan',), df['t1'].values),
            'c1': (('scan',), df['c1'].values),
            's_raw': (('scan',), df['s_raw'].values),
        },
        coords={'scan': np.arange(len(df))}
    )
    
    # Assign Geographic Metadata (Global variables for the whole profile)
    if not np.isnan(meta['lat']):
        ds = ds.assign_coords(latitude=meta['lat'])
        ds['latitude'].attrs = {
            'units': 'degrees_north',
            'standard_name': 'latitude',
            'long_name': 'Latitude'
        }
        
    if not np.isnan(meta['lon']):
        ds = ds.assign_coords(longitude=meta['lon'])
        ds['longitude'].attrs = {
            'units': 'degrees_east',
            'standard_name': 'longitude',
            'long_name': 'Longitude'
        }
        
    if meta['datetime'] is not None:
        ds = ds.assign_coords(time=meta['datetime'])
        ds['time'].attrs = {
            'standard_name': 'time',
            'long_name': 'Time',
            'axis': 'T'
        }

    # Global Attributes assignment based on CF Conventions
    ds.attrs['title'] = f"MVP Profile {filename} ({cast_type})"
    ds.attrs['source_file'] = filename
    ds.attrs['instrument'] = 'Moving Vessel Profiler'
    ds.attrs['cast_direction'] = cast_type
    ds.attrs['processing_level'] = 'Raw data converted from .m1'
    ds.attrs['history'] = f'Created {datetime.now().strftime("%Y-%m-%d")}; Split {cast_type}'
    ds.attrs['Conventions'] = 'CF-1.6'
    ds.attrs['processed_by'] = 'Elisabet Verger-Miralles'
    ds.attrs['processing_code_repository'] = 'https://github.com/everger-miralles/paper_processing_Moving_Vessel_Profiler_Data'
    ds.attrs['ship'] = 'R/V SOCIB'

    fname_lower = filename.lower()
    if ('fast-swot1' in fname_lower) or ('leg1' in fname_lower):
        ds.attrs['cruise'] = 'FaSt-SWOT Leg 1'
    elif ('fast-swot2' in fname_lower) or ('leg2' in fname_lower):
        ds.attrs['cruise'] = 'FaSt-SWOT Leg 2'
    else:
        ds.attrs['cruise'] = 'FaSt-SWOT'

    if meta['datetime'] is not None:
        dt_utc = pd.to_datetime(meta['datetime'], utc=True)
        ds.attrs['date_utc'] = dt_utc.strftime('%Y-%m-%d')
        ds.attrs['time_utc'] = dt_utc.strftime('%H:%M:%S')
    else:
        ds.attrs['date_utc'] = 'unknown'
        ds.attrs['time_utc'] = 'unknown'
    
    # Variable Attributes assignment
    ds['pressure'].attrs = {
        'units': 'dbar',
        'long_name': 'Pressure',
        'standard_name': 'sea_water_pressure',
        'axis': 'Z',
        'positive': 'down'
    }
    ds['t1'].attrs = {'units': 'degC', 'long_name': 'Temperature', 'standard_name': 'sea_water_temperature'}
    ds['c1'].attrs = {'units': 'mS/cm', 'long_name': 'Conductivity', 'standard_name': 'sea_water_electrical_conductivity'}
    ds['s_raw'].attrs = {
        'units': 'PSU',
        'long_name': 'Raw Salinity (Instrument)',
        'standard_name': 'sea_water_practical_salinity'
    }
    
    return ds


# --- 3. MAIN PROCESSING ROUTINE ---

def process_m1_file(filepath, output_dir):
    """
    Processes a single .m1 file: parses metadata, reads data, splits into 
    downcast/upcast, and saves to NetCDF.
    """
    filename = os.path.basename(filepath)
    file_stem = filename.replace('.m1', '') # Filename without extension
    
    # A) Read Metadata from Header
    meta = parse_m1_header(filepath)
    
    # B) Read Data Body
    col_names = ['pressure', 'depth', 'sv', 't1', 'c1', 's_raw', 'density', 'aux1', 'aux2']
    try:
        # Use python engine and skip bad lines to bypass weird header remnants
        df = pd.read_csv(filepath, names=col_names, comment='*', 
                         sep=r'[,\s]+', engine='python', on_bad_lines='skip')
        
        # Keep only rows where 'pressure' is a valid number
        df = df[pd.to_numeric(df['pressure'], errors='coerce').notnull()].astype(float)
        df = df.reset_index(drop=True)
        
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")
        return

    if df.empty:
        print(f"⚠️ Empty or no valid numeric data found in: {filename}")
        return

    # C) Detect Turning Point (Maximum Pressure)
    idx_max = df['pressure'].idxmax()
    
    # Separate DataFrames
    # Downcast: From beginning to Max Pressure (inclusive)
    df_down = df.iloc[0 : idx_max + 1].copy().reset_index(drop=True)
    # Upcast: From Max Pressure to end
    df_up = df.iloc[idx_max + 1 : ].copy().reset_index(drop=True)
    
    # D) Generate and Save DOWNCAST Dataset
    if not df_down.empty:
        ds_down = create_mvp_dataset(df_down, meta, filename, 'downcast')
        out_name_down = f"{file_stem}_downcast.nc"
        ds_down.to_netcdf(os.path.join(output_dir, out_name_down))
    
    # E) Generate and Save UPCAST Dataset
    if not df_up.empty:
        ds_up = create_mvp_dataset(df_up, meta, filename, 'upcast')
        out_name_up = f"{file_stem}_upcast.nc"
        ds_up.to_netcdf(os.path.join(output_dir, out_name_up))


# --- 4. EXECUTION ---

def process_directory(input_dir, output_dir, label):
    """
    Iterates over all .m1 files in the input directory.
    """
    files = glob(os.path.join(input_dir, "*.m1"))
    print(f"--- {label}: Processing {len(files)} files ---")
    
    count = 0
    for f in files:
        process_m1_file(f, output_dir)
        count += 1
        if count % 100 == 0: 
            print(f"   ... {count} files processed")
            
    print(f"✅ Completed. Output saved in: {output_dir}\n")

# Run the batch process
process_directory(RAW_DIR_LEG1, OUT_DIR_LEG1, "LEG 1")
process_directory(RAW_DIR_LEG2, OUT_DIR_LEG2, "LEG 2")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


--- LEG 1: Processing 691 files ---
   ... 100 files processed
   ... 200 files processed
   ... 300 files processed
   ... 400 files processed
   ... 500 files processed
   ... 600 files processed
✅ Completed. Output saved in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\raw

--- LEG 2: Processing 333 files ---
   ... 100 files processed
   ... 200 files processed
   ... 300 files processed
✅ Completed. Output saved in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg2\raw

